# Assignment 3 - Gemini in Colab (no GPU or model downloads)

Select **Runtime > Change runtime type > CPU**, then **Runtime > Run all**. Enter your Gemini API key only in the private prompt below. Use the **Free Tier project with billing disabled** that you created. Do not paste the key into a code cell, chat or GitHub.

The notebook runs all 50 PDFs through a lower-priced Flash-Lite model and a higher-priced Flash model using available free allowance, then a refined hybrid. Five invoices per request reduce request count; this is ordinary synchronous generation, not the asynchronous paid Batch API. Model availability and quota are checked at runtime; the model-list API does not independently verify billing status. If access or quota fails, the run stops without upgrading or choosing a paid-only model. Runtime depends on quota and service latency; no completion time is guaranteed.

Only source invoice text is sent to Google. Ground truth is used later for evaluation. Google's free-tier data-use terms apply. The published standard prices are recorded separately from the $0 charge applicable to eligible free-tier usage.

Download the final results ZIP and **File > Download > Download .ipynb**, then return both files. This notebook is an unexecuted template until run in your account.

Sources: [Pricing](https://ai.google.dev/gemini-api/docs/pricing), [Billing and free tier](https://ai.google.dev/gemini-api/docs/billing), [GenerateContent reference](https://ai.google.dev/api/generate-content).


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','pypdf>=5,<7','requests'],check=True)
print('Ready. No GPU or large model download is needed.')


In [ ]:
from pathlib import Path
import io,json,zipfile,requests,os,hashlib,datetime,importlib.metadata,shutil,csv,getpass
WORK=Path('/content/assignment3_gemini_run');WORK.mkdir(exist_ok=True)
r=requests.get('https://codeload.github.com/MVEST22/invoice-extraction-assignment/zip/refs/heads/main',timeout=120)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    for entry in z.infolist():
        p=Path(entry.filename)
        if p.is_absolute() or '..' in p.parts:raise ValueError('Unexpected source archive path')
    z.extractall(WORK)
ROOT=WORK/'invoice-extraction-assignment-main';os.chdir(ROOT)
subprocess.run([sys.executable,'scripts/setup_data.py'],check=True)
subprocess.run([sys.executable,'scripts/preflight.py'],check=True)
sys.path.insert(0,str(ROOT/'methods'))
print('50 source invoices ready; answers are reserved for evaluation.')


In [ ]:
subprocess.run([sys.executable,'methods/method1_programmatic.py'],check=True)
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)


## Private key and model access
Use your key from the Free Tier project with billing disabled. This check does not enable billing. No API key is written to files.

In [ ]:
from gemini_extract import choose_models,run
KEY=getpass.getpass('Gemini API key from your Free Tier project (private): ').strip()
if not KEY:raise RuntimeError('No key entered. Do not share the key in chat.')
MODEL_PAIR=choose_models(KEY)
print('Model 2:',MODEL_PAIR['method2_low_cost'])
print('Model 3:',MODEL_PAIR['method3_expensive'])


## Method 2 - lower-priced Gemini
Each line reports another five completed invoices. Responses and usage are saved immediately; same-runtime reruns reuse matching checkpoints.

In [ ]:
run(KEY,'method2_low_cost',MODEL_PAIR['method2_low_cost'])
print('Method 2 finished.')


## Method 3 - higher-priced Gemini using free allowance

In [ ]:
run(KEY,'method3_expensive',MODEL_PAIR['method3_expensive'])
KEY=None
print('Method 3 finished; private key cleared from the notebook variable.')


## Compare the first three methods, then refine and evaluate
The refined method reads source PDFs and candidates only. It preserves printed values and records arithmetic flags, rather than copying supplied answers.

In [ ]:
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)
shutil.copyfile(ROOT/'evaluation/summary.json',ROOT/'evaluation/initial_three_methods.json')
for script in ['methods/method4_refined.py','evaluation/compare_results.py','scripts/calculate_costs.py',
               'scripts/build_db.py','scripts/build_report.py']:
    subprocess.run([sys.executable,script],check=True)
print((ROOT/'REPORT.md').read_text())


## Download results
This final cell creates the ZIP. You can also run it after an earlier error to download partial evidence. The manifest clearly identifies incomplete runs.

In [ ]:
from google.colab import files
DEST=WORK/'results_package';DEST.mkdir(exist_ok=True)
for directory in ['outputs','evaluation','methods','scripts']:
    shutil.copytree(ROOT/directory,DEST/directory,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
for name in ['REPORT.md','README.md','requirements.txt','run_pipeline.py','tests_local.py','invoice_results.db','Assignment_3_Gemini_Colab.ipynb']:
    if (ROOT/name).exists():shutil.copyfile(ROOT/name,DEST/name)
status={}
for method in ['method1_programmatic','method2_low_cost','method3_expensive','method4_refined']:
    path=ROOT/'outputs'/(method+'.csv')
    if path.exists():
        with path.open(encoding='utf-8-sig',newline='') as f:status[method]=len(list(csv.DictReader(f)))
    else:status[method]=0
costpath=ROOT/'evaluation/costs.json'
costs=json.loads(costpath.read_text()) if costpath.exists() else []
manifest={'packaged_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'rows':status,
    'complete':all(n==50 for n in status.values()),'method':'Gemini API on CPU Colab; no local model inference',
    'models':globals().get('MODEL_PAIR',{}),'billing_basis':'Eligible Free Tier project chosen by user; billing disabled; not independently verified by inference API.',
    'free_tier_charge_usd':0,'usage':costs,
    'packages':{p:importlib.metadata.version(p) for p in ['pypdf','requests']},
    'input_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((ROOT/'invoices').glob('*.pdf'))},
    'code_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for d in ['methods','scripts','evaluation'] for p in (ROOT/d).glob('*.py')}}
(DEST/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
(DEST/'EXECUTION_NOTE.txt').write_text('The included notebook is an unexecuted source template. Download your actual executed notebook separately using File > Download > Download .ipynb. Return that notebook and this ZIP. Check manifest complete and row counts. API key is not included.')
archive=shutil.make_archive(str(WORK/'assignment3_results'),'zip',DEST)
print('Rows per method:',status,'Complete:',manifest['complete'])
files.download(archive)
